# Pillar curves

A discount curve, a forward curve, a volatility curve, and a fixing curve are the numbers a
trade is priced from. This notebook builds the discount and forward curves from the sample
market's flat rates: spot 17.50 on 16 September 2026, domestic rate 7%, foreign rate 4%.
The volatility pillars rise from 12% to 16%, so the term structure is visible on the chart.
A later section uses a flat 12% curve.

Forwards use the interest-rate differential, `spot * exp((domestic - foreign) * year fraction)`.
Discount factors use the domestic rate, `exp(-domestic * year fraction)`. Year fractions are
Actual/365. The curve then interpolates those pillar values.


In [ ]:
import logging
import math
from datetime import date, timedelta

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import matplotlib.pyplot as plt
import sjd
from sjd import examples

sjd.set_log_level("INFO")

anchor = examples.ANCHOR
pillars = [
    anchor,
    date(2026, 12, 16),
    date(2027, 6, 16),
    date(2027, 9, 16),
]
forwards, discount_factors, _ = examples.fx_curve_values(pillars)
volatilities = [0.12, 0.13, 0.15, 0.16]
built = sjd.ForwardCurve(anchor, pillars, forwards)
discount = sjd.DiscountCurve(anchor, pillars, discount_factors)
volatility = sjd.VolatilityCurve(anchor, pillars, volatilities)
from_examples = examples.fx_curves(pillars)

for day, forward, factor, vol in zip(pillars, forwards, discount_factors, volatilities):
    assert math.isclose(built.get_forward(day), forward)
    assert math.isclose(discount.get_discount_factor(day), factor)
    assert math.isclose(volatility.get_volatility(day), vol)
    assert math.isclose(from_examples[0].get_forward(day), forward)
for curve in (built, discount, volatility):
    assert curve.validate().ok

print("forward", built)
print("discount", discount)
print("volatility", volatility)


## Between the pillars

Interpolation is on the pillar values, along Actual/365. Halfway between two forward pillars
the forward is halfway between the two values. That is not the same number as the
continuously compounded formula, which is what was used to build the pillars themselves.

A cubic interpolator uses the same pillars and a different path between them.


In [ ]:
left, right = pillars[1], pillars[2]
midpoint = left + (right - left) / 2
weight = (midpoint - left).days / (right - left).days
linear_forward = built.get_forward(left) + weight * (
    built.get_forward(right) - built.get_forward(left)
)
compounded = examples.SPOT * math.exp(
    (examples.DOMESTIC_RATE - examples.FOREIGN_RATE) * examples.year_fraction(midpoint)
)

assert math.isclose(built.get_forward(midpoint), linear_forward, rel_tol=1e-12)
assert not math.isclose(built.get_forward(midpoint), compounded, rel_tol=1e-12)

cubic = sjd.ForwardCurve(
    anchor, pillars, forwards, interpolation=sjd.CurveInterpolator.CUBIC
)
assert cubic.validate().ok
assert not math.isclose(
    cubic.get_forward(midpoint), built.get_forward(midpoint), rel_tol=1e-12
)
print(f"linear {built.get_forward(midpoint):.6f}")
print(f"cubic  {cubic.get_forward(midpoint):.6f}")
print(f"formula {compounded:.6f}")


In [ ]:
plot_dates = []
day = anchor
while day <= pillars[-1]:
    plot_dates.append(day)
    day += timedelta(days=7)

figure, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].plot(plot_dates, [discount.get_discount_factor(day) for day in plot_dates])
axes[0].set_title("Discount factor")
axes[1].plot(plot_dates, [built.get_forward(day) for day in plot_dates], label="linear")
axes[1].plot(plot_dates, [cubic.get_forward(day) for day in plot_dates], label="cubic")
axes[1].set_title("Forward")
axes[1].legend()
axes[2].plot(plot_dates, [volatility.get_volatility(day) for day in plot_dates])
axes[2].set_title("Volatility")
for axis in axes:
    axis.tick_params(axis="x", rotation=30)
figure.tight_layout()
figure


## A flat volatility stays flat

The volatility curve interpolates total variance, volatility squared times the year fraction.
A flat 12% curve is therefore 12% before the first pillar as well as between the pillars.
The first pillar here is after the anchor, so the anchor itself is filled in flat.


In [ ]:
flat = sjd.VolatilityCurve(
    anchor,
    [date(2026, 12, 16), date(2027, 9, 16)],
    [examples.FLAT_VOLATILITY, examples.FLAT_VOLATILITY],
    allow_extrapolation=True,
)
flat_report = flat.validate()
before_first = date(2026, 10, 16)
between = date(2027, 3, 16)

assert flat_report.ok
assert math.isclose(flat.get_volatility(before_first), examples.FLAT_VOLATILITY, rel_tol=1e-9)
assert math.isclose(flat.get_volatility(between), examples.FLAT_VOLATILITY, rel_tol=1e-9)
assert {issue.code for issue in flat_report.issues} == {
    "RATE_SOURCE_FLAT_BACKFILL",
    "RATE_SOURCE_EXTRAPOLATION",
}
assert all(issue.severity == sjd.ValidationSeverity.WARNING for issue in flat_report.issues)
flat_report


## Calendar-month curves

With `averaging=ForwardAveraging.CALENDAR_MONTH`, the pillar date only supplies the month.
Every day in that month has the same forward. `VolatilityCurveCMA` does the same for a
volatility. A calendar-month forward cannot roll with spot volatility: that option is refused
at construction.


In [ ]:
month_starts = [date(2026, 10, 1), date(2026, 12, 1)]
monthly = sjd.ForwardCurve(
    anchor,
    month_starts,
    [17.40, 17.80],
    averaging=sjd.ForwardAveraging.CALENDAR_MONTH,
)
monthly_vol = sjd.VolatilityCurveCMA(anchor, month_starts, [0.12, 0.20])
assert monthly.validate().ok
assert monthly_vol.validate().ok
assert math.isclose(monthly.get_forward(date(2026, 10, 5)), 17.40)
assert math.isclose(monthly.get_forward(date(2026, 10, 28)), 17.40)
assert math.isclose(monthly.get_forward(date(2026, 12, 2)), 17.80)
assert math.isclose(monthly_vol.get_volatility(date(2026, 10, 5)), 0.12)
assert math.isclose(monthly_vol.get_volatility(date(2026, 12, 3)), 0.20)

try:
    sjd.ForwardCurve(
        anchor,
        month_starts,
        [17.40, 17.80],
        averaging=sjd.ForwardAveraging.CALENDAR_MONTH,
        roll=sjd.ForwardRoll.SPOT_VOLATILITY,
    )
except sjd.ConfigurationError as error:
    assert error.code == "CONFIGURATION"
    print(error)
else:
    raise AssertionError("a calendar-month curve has no spot-volatility roll")


## Fixings, rates, and a bump

A `DailyFixingCurve` holds rates that have already been set. `InterestRateConvention`
converts a discount factor and a continuously compounded Actual/365 rate.
`parallel_bump` returns a new discount curve; the original pillars are unchanged.


In [ ]:
fixings = sjd.DailyFixingCurve([date(2026, 9, 15), anchor], [17.20, examples.SPOT])
assert fixings.validate().ok
assert math.isclose(fixings.get_rate(date(2026, 9, 15)), 17.20)
assert math.isclose(fixings.get_rate(anchor), examples.SPOT)

convention = sjd.InterestRateConvention(sjd.Compounding.CONTINUOUS, sjd.Frequency.ANNUAL)
horizon = pillars[-1]
factor = convention.rate_to_df(examples.DOMESTIC_RATE, anchor, horizon)
assert math.isclose(factor, math.exp(-examples.DOMESTIC_RATE * examples.year_fraction(horizon)))
assert math.isclose(convention.df_to_rate(factor, anchor, horizon), examples.DOMESTIC_RATE)

bumped = discount.parallel_bump(0.01, convention)
assert bumped.get_discount_factor(horizon) < discount.get_discount_factor(horizon)
assert math.isclose(discount.get_discount_factor(horizon), discount_factors[-1])
print(f"discount {discount.get_discount_factor(horizon):.6f}")
print(f"bumped   {bumped.get_discount_factor(horizon):.6f}")


## What can go wrong

A curve whose dates and values have different lengths is constructed, and `validate()` reports
the interpolator's dimension code. The path runs through the curve to the interpolator.
`is_valid` and `report` read that result. Reading them again does not run the checks a second
time: the stored issue list is the one `validate()` returned.

The flat curve above is valid. Its two issues are warnings, so `report.ok` stays true.
Extrapolation past the last pillar is the second of those warnings. While extrapolation is
off, a lookup past the last pillar is outside the curve.


In [ ]:
mismatched = sjd.DiscountCurve(anchor, [anchor, date(2027, 9, 16)], [1.0])
assert mismatched.report.issues == ()

mismatch_report = mismatched.validate()
assert not mismatch_report.ok
assert not mismatched.is_valid
issue = mismatch_report.issues[0]
assert issue.code == "INTERPOLATOR_DIMENSION_MISMATCH"
assert issue.path == ("discount_curve", "dated_curve", "interpolator")
assert [item.code for item in mismatched.report.issues] == [issue.code]
mismatch_report


## What it logs

The same two warnings are also `WARNING` records on `sjd._core`. Validating the flat curve
again produces one record for the flat anchor and one for the open extrapolation.


In [ ]:
class Collect(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []

    def emit(self, record):
        self.records.append(record)

captured = Collect()
logging.getLogger("sjd._core").addHandler(captured)
logged = flat.validate()
messages = [record.getMessage() for record in captured.records]

assert logged.ok
assert any("flat through the first observation" in message for message in messages)
assert any("Extrapolation past the last pillar" in message for message in messages)
assert all(record.levelno == logging.WARNING for record in captured.records)
for message in messages:
    print("WARNING", message)


## Save and reload

Each curve pickles from the pillars it was built with. The restored curve reads the same
values. The file is in a temporary directory.


In [ ]:
import pickle
import tempfile
from pathlib import Path

scratch = tempfile.TemporaryDirectory()
path = Path(scratch.name) / "curves.pkl"
path.write_bytes(pickle.dumps((built, discount, volatility, fixings)))
restored_forward, restored_discount, restored_volatility, restored_fixings = pickle.loads(
    path.read_bytes()
)

assert math.isclose(restored_forward.get_forward(midpoint), built.get_forward(midpoint))
assert math.isclose(
    restored_discount.get_discount_factor(horizon), discount.get_discount_factor(horizon)
)
assert math.isclose(
    restored_volatility.get_volatility(horizon), volatility.get_volatility(horizon)
)
assert math.isclose(restored_fixings.get_rate(anchor), examples.SPOT)
print("reloaded", path.name)
